<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


In [2]:
!git clone --depth 1 https://github.com/rasbt/LLMs-from-scratch.git

fatal: destination path 'LLMs-from-scratch' already exists and is not an empty directory.


- Install the additional package requirements for this bonus notebook by uncommenting and running the following cell:

In [3]:
# pip install -r requirements-extra.txt

# Comparing Various Byte Pair Encoding (BPE) Implementations

<br>
&nbsp;

## 1. Using BPE from `tiktoken`

In [4]:
from importlib.metadata import version

print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.12.0


In [5]:
import tiktoken

tik_tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello, world. Is this-- a test?"

In [6]:
integers = tik_tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]


In [7]:
strings = tik_tokenizer.decode(integers)

print(strings)

Hello, world. Is this-- a test?


In [8]:
print(tik_tokenizer.n_vocab)

50257


<br>
&nbsp;

## 2. Using the original BPE implementation used in GPT-2

In [9]:
!find LLMs-from-scratch -iname "bpe_openai_gpt2.py"

LLMs-from-scratch/ch02/02_bonus_bytepair-encoder/bpe_openai_gpt2.py


In [10]:
# from bpe_openai_gpt2 import get_encoder, download_vocab
import sys, os

repo_path = "LLMs-from-scratch/ch02/02_bonus_bytepair-encoder"
sys.path.append(repo_path)

from bpe_openai_gpt2 import get_encoder, download_vocab

In [11]:
download_vocab()

Fetching encoder.json: 1.04Mit [00:00, 2.19Mit/s]                                                   
Fetching vocab.bpe: 457kit [00:00, 1.71Mit/s]                                                       


In [12]:
orig_tokenizer = get_encoder(model_name="gpt2_model", models_dir=".")

In [13]:
integers = orig_tokenizer.encode(text)

print(integers)

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]


In [14]:
strings = orig_tokenizer.decode(integers)

print(strings)

Hello, world. Is this-- a test?


<br>
&nbsp;

## 3. Using the BPE via Hugging Face transformers

In [15]:
import transformers

transformers.__version__

'5.0.0'

In [16]:
from transformers import GPT2Tokenizer

hf_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

In [17]:
hf_tokenizer(strings)["input_ids"]

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]

In [18]:
from transformers import GPT2TokenizerFast

hf_tokenizer_fast = GPT2TokenizerFast.from_pretrained("gpt2")

In [19]:
hf_tokenizer_fast(strings)["input_ids"]

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]

<br>
&nbsp;

## 4. Using my own from-scratch BPE tokenizer

In [20]:
import os
import sys
import io
import nbformat
import types

def import_from_notebook():
    def import_definitions_from_notebook(fullname, names):
        current_dir = os.getcwd()
        # path = os.path.join(current_dir, "..", "05_bpe-from-scratch", fullname + ".ipynb")
        path = os.path.join("LLMs-from-scratch", "ch02", "05_bpe-from-scratch", fullname + ".ipynb")
        path = os.path.normpath(path)

        # Load the notebook
        if not os.path.exists(path):
            raise FileNotFoundError(f"Notebook file not found at: {path}")

        with io.open(path, "r", encoding="utf-8") as f:
            nb = nbformat.read(f, as_version=4)

        # Create a module to store the imported functions and classes
        mod = types.ModuleType(fullname)
        sys.modules[fullname] = mod

        # Go through the notebook cells and only execute function or class definitions
        for cell in nb.cells:
            if cell.cell_type == "code":
                cell_code = cell.source
                for name in names:
                    # Check for function or class definitions
                    if f"def {name}" in cell_code or f"class {name}" in cell_code:
                        exec(cell_code, mod.__dict__)
        return mod

    fullname = "bpe-from-scratch"
    names = ["BPETokenizerSimple"]

    return import_definitions_from_notebook(fullname, names)

In [21]:
imported_module = import_from_notebook()
BPETokenizerSimple = getattr(imported_module, "BPETokenizerSimple", None)

tokenizer_gpt2 = BPETokenizerSimple()
tokenizer_gpt2.load_vocab_and_merges_from_openai(
    vocab_path=os.path.join("gpt2_model", "encoder.json"),
    bpe_merges_path=os.path.join("gpt2_model", "vocab.bpe")
)

In [22]:
integers = tokenizer_gpt2.encode(text)

print(integers)

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]


<br>
&nbsp;

## 5. A quick performance benchmark

In [23]:
!find LLMs-from-scratch -iname "the-verdict.txt"

LLMs-from-scratch/appendix-D/01_main-chapter-code/the-verdict.txt
LLMs-from-scratch/ch05/05_bonus_hparam_tuning/the-verdict.txt
LLMs-from-scratch/ch02/01_main-chapter-code/the-verdict.txt


In [24]:
verdict_file = "/kaggle/working/LLMs-from-scratch/ch02/01_main-chapter-code/the-verdict.txt"

In [25]:
with open(verdict_file, "r", encoding="utf-8") as f:
    raw_text = f.read()

&nbsp;
### 5.1 Original OpenAI GPT-2 tokenizer

In [26]:
%timeit orig_tokenizer.encode(raw_text)

12 ms ± 120 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


&nbsp;
### 5.2 Tiktoken OpenAI GPT-2 tokenizer

In [27]:
%timeit tik_tokenizer.encode(raw_text)

2.43 ms ± 27.3 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


&nbsp;
### 5.3 Hugging Face OpenAI GPT-2 tokenizer

In [28]:
%timeit hf_tokenizer(raw_text)["input_ids"]

Token indices sequence length is longer than the specified maximum sequence length for this model (5145 > 1024). Running this sequence through the model will result in indexing errors


13.2 ms ± 293 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [29]:
%timeit hf_tokenizer(raw_text, max_length=5145, truncation=True)["input_ids"]

13.2 ms ± 229 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [30]:
%timeit hf_tokenizer_fast(raw_text)["input_ids"]

Token indices sequence length is longer than the specified maximum sequence length for this model (5145 > 1024). Running this sequence through the model will result in indexing errors


14.1 ms ± 145 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [31]:
%timeit hf_tokenizer_fast(raw_text, max_length=5145, truncation=True)["input_ids"]

14.2 ms ± 197 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


&nbsp;
### 5.4 My own GPT-2 tokenizer (for educational purposes)

In [32]:
%timeit tokenizer_gpt2.encode(raw_text)

38.7 ms ± 1.34 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
